# Systematics + detector variations

Continues from `step2_combining_mc_and_data.ipynb` -- run that first.
Covers adding systematics and detector variations, plus a final
conformance check before shipping a new analysis.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../../../..")))

import pandas as pd
import cafpybara.analyses._template as ana
from cafpybara import core

print("Imported cleanly -- a real, importable (if inert) analysis.")

## 1. `funcs.py` -- adding systematics

- Error bands come from `systs=` -- a `SystematicsInput` built by
  `funcs.py`'s `systs_input()`, then unpacked into
  `core.funcs.get_total_cov`.
- Rule: `core.funcs.get_total_cov` is the *only* `get_total_cov` -- no
  analysis defines its own. `systs_input(...)` resolves every default
  into a concrete `SystematicsInput` before anything reaches `core`, so
  every caller necessarily agrees.
- `uncertainty_keys` picks which sources go in; leave out `'detv'` for
  now.

```python
si = ana.systs_input(pot, uncertainty_keys={'rate', 'norm'})
ana.plot_mc_data(mc_df, data_df, "your_variable", bins=[0, 1, 2, 3], systs=si)
```

In [ ]:
print("ana.get_total_cov IS core.funcs.get_total_cov:", ana.get_total_cov is core.funcs.get_total_cov)
print("(if this were ever False for a new analysis, that's the rule above broken)\n")

si = ana.systs_input(1e20, uncertainty_keys={'rate', 'norm'}, detvar_dict={'placeholder': True})
print("systs_input() resolved cuts to DEFAULT_CUTS:", si.cuts is ana.DEFAULT_CUTS)
print("resolved uncertainty_keys:", si.uncertainty_keys)
print("\npass si directly as plot_var(..., systs=si), or si.to_kwargs() into get_total_cov:")
print(list(si.to_kwargs().keys()))

## 2. `funcs.py` + `config.py` -- adding detector variations

- Add `'detv'` to `uncertainty_keys` to fold detector-systematic
  uncertainty into the same covariance matrix.
- Needs `detvar_dict=`/`detvar_files=` -- `systs_input()` defaults
  `detvar_files` to `config.py`'s `DETVAR_DICT_FILES` if you don't pass one.
- `config.py` is path constants only. `DETVAR_DICT_FILES` is an
  obviously-fake path, not `None` -- fails loud (`FileNotFoundError`) if
  used unfilled. `INTIME_FILE` is the other constant here, for cosmic
  subtraction (opt in via `'cosmic'` in `uncertainty_keys`, not covered above).
- Building real detvar stores: add an entry for your analysis in
  `core/detvar/process_detvars.py`'s `_SLC_KEY`/`_default_preprocess_fn`/
  `_selection_fn_map` -- the one place in `core/` that needs an edit.

```python
si = ana.systs_input(pot, uncertainty_keys={'rate', 'norm', 'detv'})
ana.plot_mc_data(mc_df, data_df, "your_variable", bins=[0, 1, 2, 3], systs=si)
```

In [ ]:
print("DETVAR_DICT_FILES:", ana.config.DETVAR_DICT_FILES)
print("INTIME_FILE:", ana.config.INTIME_FILE)

si = ana.systs_input(1e20, uncertainty_keys={'rate', 'norm', 'detv'})
print("\nleaving detvar_dict/detvar_files unset falls back to config.py:")
print("si.detvar_files is ana.config.DETVAR_DICT_FILES:", si.detvar_files is ana.config.DETVAR_DICT_FILES)

## 3. Before you ship it: a manual conformance pass

`scripts/verify_new_analysis.py <name>` runs this automatically (plus two
checks this manual version can't: a no-op-preprocessing-default audit, and
an ast-based sweep of every example notebook's `systs_input`/
`SystematicsInput`/`PlottingConfig` calls for dead kwargs). If your analysis
has a loader that needs real preprocessing, add an entry for it to that
script's `NOOP_DEFAULT_CHECKS` table -- it's curated per-analysis, not
derived automatically. The version below is the same idea as the script's
first two checks, inline, so you can see how it works without leaving the
notebook:

Fuller checklist once this passes: README.md's "Reference: the override
contract" and "Common mistakes" sections.

In [ ]:
def conformance_check(analysis_module):
    """Run this against your real analysis module, not the template."""
    problems = []

    for mod_name in ['io', 'funcs', 'plotting', 'selection', 'physics', 'syst', 'classes', 'preprocess', 'detvar']:
        core_mod = getattr(core, mod_name)
        missing = [n for n in getattr(core_mod, '__all__', []) if not hasattr(analysis_module, n)]
        if missing:
            problems.append(f"core.{mod_name} not fully re-exported: missing {missing}")

    if analysis_module.get_total_cov is not core.funcs.get_total_cov:
        problems.append("get_total_cov has been overridden -- delete it, keep only systs_input()")

    if not hasattr(analysis_module, 'systs_input'):
        problems.append("no systs_input() factory found")

    if problems:
        print(f"{len(problems)} problem(s) found:")
        for p in problems:
            print(f"  - {p}")
    else:
        print("No problems found by this (non-exhaustive) check.")
    return problems


conformance_check(ana)